In [0]:
from pyspark.sql import functions as F
import re

base = "/Volumes/workspace/bronze/landing"
target = "workspace.bronze"

def add_audit(df):
    return (df.withColumn("_source_file", F.col("_metadata.file_path"))
              .withColumn("_ingested_at", F.current_timestamp()))

def save(df, name):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{target}.{name}"))
    print(f"{name}: {spark.table(f'{target}.{name}').count()} rows")

In [0]:
for t in ["stores", "customers", "orders", "order_items"]:
    df = (spark.read.option("recursiveFileLookup", "true")
          .parquet(f"{base}/sqlserver/{t}"))
    save(add_audit(df), t)

stores: 10 rows
customers: 200 rows
orders: 4000 rows
order_items: 12001 rows


In [0]:
df = (spark.read.option("recursiveFileLookup", "true")
      .json(f"{base}/fakestore_api/products"))
save(add_audit(df), "products")

products: 20 rows


In [0]:
df = (spark.read.option("header", "true")
      .option("inferSchema", "false")          # keep everything as raw strings
      .csv(f"{base}/inventory_csv/*.csv"))

# Delta rejects spaces and special characters in column names
df = df.toDF(*[re.sub(r"[ ,;{}()\n\t=]+", "_", c.strip()) for c in df.columns])
df.printSchema()
save(add_audit(df), "inventory")

root
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- stock_qty: string (nullable = true)
 |-- reorder_level: string (nullable = true)
 |-- snapshot_date: string (nullable = true)

inventory: 1428 rows


In [0]:
display(spark.sql("SHOW TABLES IN workspace.bronze"))
display(spark.table("workspace.bronze.orders").limit(3))

database,tableName,isTemporary
bronze,customers,false
bronze,inventory,false
bronze,order_items,false
bronze,orders,false
bronze,products,false
bronze,stores,false


order_id,customer_id,store_id,order_date,payment_mode,order_status,updated_at,_source_file,_ingested_at
1,8,2,2026-11-24T15:29:25.602Z,Card,Completed,2026-11-24T16:32:25.602Z,dbfs:/Volumes/workspace/bronze/landing/sqlserver/orders/2026/11/25/orders.parquet,2026-11-25T17:51:31.281Z
2,15,3,2026-11-24T14:26:25.602Z,Cash,Completed,2026-11-24T16:32:25.602Z,dbfs:/Volumes/workspace/bronze/landing/sqlserver/orders/2026/11/25/orders.parquet,2026-11-25T17:51:31.281Z
3,22,4,2026-11-24T13:23:25.602Z,Wallet,Completed,2026-11-24T16:32:25.602Z,dbfs:/Volumes/workspace/bronze/landing/sqlserver/orders/2026/11/25/orders.parquet,2026-11-25T17:51:31.281Z


In [0]:
from pyspark.sql import functions as F
inv = spark.table("workspace.bronze.inventory")

# 1. Rows per source file
display(inv.groupBy("_source_file").count().orderBy("_source_file"))

# 2. Duplicate store/product/date combos
display(inv.groupBy("store_id", "product_id", "snapshot_date")
           .count().filter("count > 1"))

# 3. Nulls, blanks and non-numeric quantities
display(inv.filter(
    F.col("stock_qty").isNull() | ~F.col("stock_qty").rlike(r"^-?\d+$") |
    ~F.col("reorder_level").rlike(r"^\d+$") | F.col("store_id").isNull() |
    F.col("product_id").isNull()
))

# 4. Negative stock
display(inv.filter(F.col("stock_qty").cast("int") < 0))

_source_file,count
dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261118.csv,204
dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261119.csv,204
dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261120.csv,204
dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261121.csv,204
dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261122.csv,204
dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261123.csv,204
dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261124.csv,204


store_id,product_id,snapshot_date,count
1,1,2026-11-18,2
2,4,2026-11-18,2
3,5,2026-11-18,2
1,1,2026-11-19,2
2,4,2026-11-19,2
3,5,2026-11-19,2
1,1,2026-11-20,2
2,4,2026-11-20,2
3,5,2026-11-20,2
1,1,2026-11-21,2


store_id,product_id,stock_qty,reorder_level,snapshot_date,_source_file,_ingested_at
2,4,null,20,2026-11-18,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261118.csv,2026-11-25T17:52:00.208Z
2,4,null,20,2026-11-19,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261119.csv,2026-11-25T17:52:00.208Z
2,4,null,20,2026-11-20,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261120.csv,2026-11-25T17:52:00.208Z
2,4,null,20,2026-11-21,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261121.csv,2026-11-25T17:52:00.208Z
2,4,null,20,2026-11-22,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261122.csv,2026-11-25T17:52:00.208Z
2,4,null,20,2026-11-23,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261123.csv,2026-11-25T17:52:00.208Z
2,4,null,20,2026-11-24,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261124.csv,2026-11-25T17:52:00.208Z


store_id,product_id,stock_qty,reorder_level,snapshot_date,_source_file,_ingested_at
3,5,-15,20,2026-11-18,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261118.csv,2026-11-25T17:52:00.208Z
3,5,-15,20,2026-11-19,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261119.csv,2026-11-25T17:52:00.208Z
3,5,-15,20,2026-11-20,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261120.csv,2026-11-25T17:52:00.208Z
3,5,-15,20,2026-11-21,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261121.csv,2026-11-25T17:52:00.208Z
3,5,-15,20,2026-11-22,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261122.csv,2026-11-25T17:52:00.208Z
3,5,-15,20,2026-11-23,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261123.csv,2026-11-25T17:52:00.208Z
3,5,-15,20,2026-11-24,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261124.csv,2026-11-25T17:52:00.208Z


In [0]:
from pyspark.sql import functions as F
inv = spark.table("workspace.bronze.inventory")

# The duplicate pairs on one date, side by side
display(inv.filter(F.col("snapshot_date") == "2026-11-18")
           .filter(((F.col("store_id")=="1") & (F.col("product_id")=="1")) |
                   ((F.col("store_id")=="2") & (F.col("product_id")=="4")) |
                   ((F.col("store_id")=="3") & (F.col("product_id")=="5")))
           .orderBy("store_id", "product_id"))

# Keys outside the expected range (orphans)
display(inv.filter(F.col("snapshot_date") == "2026-11-18")
           .filter(~F.col("store_id").cast("int").between(1, 10) |
                   ~F.col("product_id").cast("int").between(1, 20)))

# Distinct keys vs total rows for that date
d = inv.filter(F.col("snapshot_date") == "2026-11-18")
print("rows:", d.count(), "distinct keys:", d.select("store_id", "product_id").distinct().count())

store_id,product_id,stock_qty,reorder_level,snapshot_date,_source_file,_ingested_at
1,1,163,10,2026-11-18,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261118.csv,2026-11-25T17:52:00.208Z
1,1,163,10,2026-11-18,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261118.csv,2026-11-25T17:52:00.208Z
2,4,null,20,2026-11-18,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261118.csv,2026-11-25T17:52:00.208Z
2,4,26,10,2026-11-18,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261118.csv,2026-11-25T17:52:00.208Z
3,5,179,30,2026-11-18,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261118.csv,2026-11-25T17:52:00.208Z
3,5,-15,20,2026-11-18,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261118.csv,2026-11-25T17:52:00.208Z


store_id,product_id,stock_qty,reorder_level,snapshot_date,_source_file,_ingested_at
99,7,50,20,2026-11-18,dbfs:/Volumes/workspace/bronze/landing/inventory_csv/inventory_20261118.csv,2026-11-25T17:52:00.208Z


rows: 204 distinct keys: 201
